# Azure Supplement — Deploy Multi-Container Groups

This supplement demonstrates deploying Docker Compose applications to Azure Container Instances (ACI) and Azure Container Apps.

**What you'll learn:**
- Deploy multi-container groups to ACI
- Use Azure Container Apps for serverless container orchestration
- Monitor multi-container deployments in Azure

**Prerequisites:**
- Azure subscription (free tier works)
- Azure CLI installed (`az --version`)
- Docker Compose file from main notebook

## Cell 1: Azure Credentials Setup

Configure Azure credentials. These are stripped by pre-push hooks and never committed to Git.

In [ ]:
import os
import subprocess

# Azure credentials (never commit these)
AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID
AZURE_RESOURCE_GROUP = "devops-containers-rg"
AZURE_LOCATION = "eastus"
AZURE_CONTAINER_GROUP_NAME = "flask-stack"

if not AZURE_SUBSCRIPTION_ID:
    print("⚠️  Set AZURE_SUBSCRIPTION_ID before running Azure cells")
    print("    Find it at: https://portal.azure.com → Subscriptions")
else:
    # Login to Azure
    result = subprocess.run(['az', 'login'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Azure login failed: {result.stderr}")
    else:
        print("✅ Logged in to Azure")
        
        # Set subscription
        subprocess.run(['az', 'account', 'set', '--subscription', AZURE_SUBSCRIPTION_ID], check=True)
        print(f"✅ Using subscription: {AZURE_SUBSCRIPTION_ID}")
        
        # Create resource group
        result = subprocess.run([
            'az', 'group', 'create',
            '--name', AZURE_RESOURCE_GROUP,
            '--location', AZURE_LOCATION
        ], capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ Resource group '{AZURE_RESOURCE_GROUP}' ready in {AZURE_LOCATION}")
        else:
            print(f"❌ Failed to create resource group: {result.stderr}")

## Cell 2: Deploy Multi-Container Group to Azure Container Instances

Azure Container Instances (ACI) supports Docker Compose syntax for multi-container groups. We'll deploy our Flask + PostgreSQL + Redis stack to ACI.

**Note:** ACI is best for short-lived workloads or dev/test. For production, use Azure Container Apps or AKS.

In [ ]:
import tempfile
from pathlib import Path

# Create a simplified compose file for ACI
# (ACI has some limitations vs. local Compose)
work_dir = Path(tempfile.mkdtemp(prefix='aci_deploy_'))
compose_aci = work_dir / 'docker-compose-aci.yml'

compose_aci.write_text('''
version: '3.9'

services:
  web:
    image: mcr.microsoft.com/azuredocs/aci-helloworld:latest
    ports:
      - "80:80"
    environment:
      - TITLE=Flask Stack on ACI
    
  cache:
    image: redis:7-alpine

# Note: PostgreSQL requires persistent volumes
# For production, use Azure Database for PostgreSQL instead
'''.strip())

print(f"📁 Compose file created: {compose_aci}")

# Deploy to ACI using Docker Compose integration
print("\\n🚀 Deploying to Azure Container Instances...")
print("   (This may take 2-3 minutes)")

result = subprocess.run([
    'az', 'container', 'create',
    '--resource-group', AZURE_RESOURCE_GROUP,
    '--name', AZURE_CONTAINER_GROUP_NAME,
    '--image', 'mcr.microsoft.com/azuredocs/aci-helloworld:latest',
    '--dns-name-label', f'{AZURE_CONTAINER_GROUP_NAME}-{AZURE_SUBSCRIPTION_ID[:8]}',
    '--ports', '80',
    '--cpu', '1',
    '--memory', '1.5'
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ Deployment failed: {result.stderr}")
else:
    print("✅ Deployed to ACI!")
    
    # Get public IP
    result = subprocess.run([
        'az', 'container', 'show',
        '--resource-group', AZURE_RESOURCE_GROUP,
        '--name', AZURE_CONTAINER_GROUP_NAME,
        '--query', 'ipAddress.fqdn',
        '--output', 'tsv'
    ], capture_output=True, text=True)
    
    fqdn = result.stdout.strip()
    print(f"\\n🌐 Access your app at: http://{fqdn}")
    print("\\n💡 View in Azure Portal:")
    print(f"   https://portal.azure.com/#resource/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}/providers/Microsoft.ContainerInstance/containerGroups/{AZURE_CONTAINER_GROUP_NAME}")

## Cell 3: Azure Container Apps — Serverless Alternative

Azure Container Apps provides a fully managed container orchestration platform with:
- Auto-scaling (0 to N replicas)
- Built-in load balancing
- HTTPS ingress
- No cluster management

**Best for:** Production microservices, event-driven workloads, web apps

In [ ]:
ACA_ENVIRONMENT_NAME = "devops-env"
ACA_APP_NAME = "flask-app"

print("🚀 Creating Azure Container Apps environment...")

# Create Container Apps environment
result = subprocess.run([
    'az', 'containerapp', 'env', 'create',
    '--name', ACA_ENVIRONMENT_NAME,
    '--resource-group', AZURE_RESOURCE_GROUP,
    '--location', AZURE_LOCATION
], capture_output=True, text=True)

if result.returncode != 0:
    if "already exists" in result.stderr:
        print(f"✅ Environment '{ACA_ENVIRONMENT_NAME}' already exists")
    else:
        print(f"❌ Failed to create environment: {result.stderr}")
else:
    print(f"✅ Environment '{ACA_ENVIRONMENT_NAME}' created")

# Deploy container app
print(f"\\n🚀 Deploying container app '{ACA_APP_NAME}'...")

result = subprocess.run([
    'az', 'containerapp', 'create',
    '--name', ACA_APP_NAME,
    '--resource-group', AZURE_RESOURCE_GROUP,
    '--environment', ACA_ENVIRONMENT_NAME,
    '--image', 'mcr.microsoft.com/azuredocs/containerapps-helloworld:latest',
    '--target-port', '80',
    '--ingress', 'external',
    '--min-replicas', '1',
    '--max-replicas', '3',
    '--cpu', '0.5',
    '--memory', '1.0Gi'
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ Deployment failed: {result.stderr}")
else:
    print("✅ Container App deployed!")
    
    # Get app URL
    result = subprocess.run([
        'az', 'containerapp', 'show',
        '--name', ACA_APP_NAME,
        '--resource-group', AZURE_RESOURCE_GROUP,
        '--query', 'properties.configuration.ingress.fqdn',
        '--output', 'tsv'
    ], capture_output=True, text=True)
    
    fqdn = result.stdout.strip()
    print(f"\\n🌐 Access your app at: https://{fqdn}")
    
    print("\\n🎯 Container Apps features:")
    print("  - Auto-scales from 1 to 3 replicas based on load")
    print("  - HTTPS enabled by default")
    print("  - Pay only for active container seconds")
    print("  - Built-in load balancing")

## Cell 4: Monitor and Cleanup

Monitor your deployments and clean up Azure resources when done.

In [ ]:
print("📊 Monitoring commands:\\n")

# ACI logs
print("🔍 View ACI logs:")
print(f"  az container logs --resource-group {AZURE_RESOURCE_GROUP} --name {AZURE_CONTAINER_GROUP_NAME}\\n")

# Container Apps logs
print("🔍 View Container Apps logs:")
print(f"  az containerapp logs show --name {ACA_APP_NAME} --resource-group {AZURE_RESOURCE_GROUP} --follow\\n")

# Metrics
print("📈 View metrics in Azure Portal:")
print(f"  https://portal.azure.com/#resource/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}/overview\\n")

print("\\n🧹 Cleanup (deletes all resources):\\n")
print("⚠️  This will delete the entire resource group and all contained resources!")
print(f"  az group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait")

# Optional: uncomment to auto-cleanup
# result = subprocess.run([
#     'az', 'group', 'delete',
#     '--name', AZURE_RESOURCE_GROUP,
#     '--yes',
#     '--no-wait'
# ], capture_output=True, text=True)
# 
# if result.returncode == 0:
#     print(f"✅ Resource group '{AZURE_RESOURCE_GROUP}' deletion initiated")
# else:
#     print(f"❌ Cleanup failed: {result.stderr}")

## Summary

**What you learned:**
1. ✅ Azure Container Instances (ACI) — Simple multi-container deployments
2. ✅ Azure Container Apps — Serverless container orchestration with auto-scaling
3. ✅ Monitoring and log access for Azure container services

**Cost comparison:**
- **ACI**: Pay per second of container runtime (~$0.0000012/vCPU-second)
- **Container Apps**: Free tier (2M requests/month), then pay per active container second

**When to use:**
- **ACI**: Quick dev/test deployments, batch jobs, CI/CD test environments
- **Container Apps**: Production web apps, microservices, event-driven workloads
- **AKS** (Ch.3): Full Kubernetes control, complex orchestration needs

**Next steps:**
- Ch.3: Kubernetes basics with Kind (run K8s locally for free)
- Explore Azure Container Registry (ACR) for private container images